# Summary_Day4.ipynb  
## AI 기초 · 신경망 · 역전파 · 활성화 함수

이번 4강은 드디어 “모델이 실제로 어떻게 학습하는가”를 코드로 보는 강의다.

앞 차시에서 Tensor와 `backward()`를 배웠다면, 이번에는 그걸 가지고 실제 머신러닝 학습 루프를 만든다.  
강의에서 계속 강조한 흐름은 다음이다.

```text
문제 정의 → 데이터 준비 → 예측 계산 → 손실 계산 → 경사 계산 → 파라미터 수정 → 반복 → 평가
```

PyTorch 코드로 보면 거의 항상 이 구조다.

```text
Forward → Loss → Backward → Update
```

이번 강의의 핵심은 다음이다.

1. AI / ML / DL 관계를 정리한다.
2. 지도학습, 비지도학습, 자기지도학습, 강화학습 차이를 정리한다.
3. 신장으로 체중을 예측하는 선형회귀 문제를 정의한다.
4. `Yp = W * X + B` 형태의 예측 함수를 만든다.
5. MSE 손실 함수를 직접 만든다.
6. `loss.backward()`로 W와 B의 gradient를 계산한다.
7. `torch.no_grad()` 안에서 파라미터를 직접 수정한다.
8. 반복 학습 루프를 만든다.
9. `optim.SGD`와 `optimizer.step()`으로 업데이트를 자동화한다.
10. `momentum`을 추가했을 때 학습이 어떻게 달라지는지 본다.
11. 활성화 함수가 왜 필요한지 정리한다.
12. Bias-Variance, Train/Validation/Test 분할, 데이터 누수 개념을 잡는다.

> 필기 포인트:  
> 이번 강의부터는 “코드가 왜 이 순서로 생겼는지”가 중요하다.  
> 단순히 외우면 헷갈리고, `예측 → 손실 → 경사 → 수정`으로 보면 거의 다 연결된다.

## 1. 라이브러리 준비

이번 실습에서는 NumPy, Matplotlib, PyTorch, scikit-learn을 사용한다.

### 함수/모듈 사용법

```python
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
```

- `np`: 숫자 배열을 만들고 계산할 때 사용한다.
- `plt`: 그래프를 그릴 때 사용한다.
- `torch`: Tensor와 자동 미분을 사용할 때 필요하다.
- `nn`: 신경망 Layer와 손실 함수를 만들 때 사용한다.
- `optim`: Optimizer를 사용할 때 사용한다.
- `train_test_split`: 데이터를 train/validation/test로 나눌 때 사용한다.

> 실습 메모:  
> 원본 노트북에는 Colab 폰트 설치, torchviz 설치, 계산 그래프 시각화 코드가 있다.  
> 여기서는 로컬에서도 바로 실행되도록 외부 설치가 필요한 코드는 빼고 핵심 학습 흐름만 정리한다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import math

from sklearn.model_selection import train_test_split

%matplotlib inline

np.set_printoptions(suppress=True, precision=4)
torch.manual_seed(123)
np.random.seed(123)

plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.unicode_minus"] = False

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)

## 2. AI / ML / DL 관계 정리

강의 자료에서는 관계를 다음처럼 정리했다.

```text
AI ⊃ ML ⊃ DL
```

- AI는 가장 넓은 개념이다.
- ML은 데이터로부터 규칙을 학습하는 AI다.
- DL은 여러 층의 신경망으로 복잡한 패턴을 학습하는 ML이다.

> 기억할 점:  
> 딥러닝은 머신러닝의 한 종류이고, 머신러닝은 인공지능의 한 종류다.

In [ ]:
concepts = {
    "AI": "Artificial Intelligence, 사람처럼 판단하거나 행동하는 넓은 기술",
    "ML": "Machine Learning, 데이터로부터 규칙을 학습하는 AI",
    "DL": "Deep Learning, 여러 층의 신경망으로 복잡한 패턴을 학습하는 ML"
}

for key, value in concepts.items():
    print(f"{key}: {value}")

## 3. 학습 방식 네 가지

강의에서는 AI가 학습하는 방식을 네 가지로 나눴다.

| 학습 방식 | 핵심 |
|---|---|
| 지도학습 | 입력과 정답이 함께 있다 |
| 비지도학습 | 정답 없이 패턴을 찾는다 |
| 자기지도학습 | 데이터 자체에서 문제와 정답을 만든다 |
| 강화학습 | 행동 후 보상을 받으며 학습한다 |

이번 실습은 **지도학습**이다.

```text
입력 X = 신장
정답 Y = 체중
```

> 필기 포인트:  
> 문제를 먼저 정의해야 데이터, 모델, 손실 함수, 평가 지표가 결정된다.

In [ ]:
learning_types = [
    ("지도학습", "입력과 정답이 함께 주어진 데이터로 학습한다"),
    ("비지도학습", "정답 없이 데이터 안의 숨은 구조를 찾는다"),
    ("자기지도학습", "데이터 자체에서 문제와 정답을 만들어 학습한다"),
    ("강화학습", "행동 후 보상을 받으면서 좋은 행동을 학습한다")
]

for name, desc in learning_types:
    print(f"{name}: {desc}")

## 4. 머신러닝 문제 정의

이번 문제는 신장으로 체중을 예측하는 단순 회귀 문제다.

```text
입력 X: 신장
정답 Y: 체중
목표: 신장과 체중의 관계를 가장 잘 설명하는 직선을 찾는 것
```

선형회귀의 기본 수식은 다음이다.

```text
Yp = W * X + B
```

- `Yp`: 예측값이다.
- `W`: weight, 기울기다.
- `B`: bias, 절편이다.
- `X`: 입력 데이터다.

> 강의식으로 기억하면:  
> 모든 딥러닝의 시작은 선형 모델이다.  
> CNN, Transformer도 결국 Linear + Activation 구조를 반복해서 쌓은 형태로 볼 수 있다.

In [ ]:
def linear_model(X, W, B):
    return W * X + B

sample_X = torch.tensor([1.0, 2.0, 3.0])
sample_W = torch.tensor(2.0)
sample_B = torch.tensor(1.0)

sample_Yp = linear_model(sample_X, sample_W, sample_B)

print(sample_Yp)

### 함수 사용법: 직접 만든 `linear_model()`

```python
linear_model(X, W, B)
```

- `X`: 입력값이다.
- `W`: 기울기 역할을 하는 파라미터다.
- `B`: 절편 역할을 하는 파라미터다.
- 반환값은 `W * X + B`다.

실습에서는 나중에 이 구조를 `pred()` 함수로 다시 만든다.

## 5. 경사하강법 큰 흐름

경사하강법은 Loss를 줄이는 방향으로 조금씩 이동하는 방법이다.

강의에서는 산 내려오기 비유로 설명했다.

```text
현재 고도 = Loss
발의 기울기 = Gradient
보폭 = Learning Rate
```

정답을 한 번에 찾는 것이 아니라, 현재 위치의 기울기를 보고 조금씩 내려간다.

학습 4단계는 다음이다.

```text
1. 예측 계산
2. 손실 계산
3. 경사 계산
4. 파라미터 수정
```

In [ ]:
gd_steps = [
    "1. 예측 계산: 현재 W와 B로 Yp를 계산한다",
    "2. 손실 계산: Yp와 Y의 차이를 Loss로 계산한다",
    "3. 경사 계산: loss.backward()로 W.grad와 B.grad를 구한다",
    "4. 파라미터 수정: W와 B를 Loss가 줄어드는 방향으로 수정한다"
]

for step in gd_steps:
    print(step)

## 6. 손실 지형과 경사하강법 시뮬레이션

원본 실습에는 3D 손실 지형 위에서 경사하강법이 내려가는 예시가 있었다.

여기서는 간단한 손실 함수 `L(u, v)`를 만들고,  
현재 위치에서 gradient 방향을 따라 파라미터가 이동하는 모습을 시뮬레이션한다.

In [ ]:
def L(u, v):
    return 3 * u**2 + 3 * v**2 - u * v + 7 * u - 7 * v + 10

def Lu(u, v):
    return 6 * u - v + 7

def Lv(u, v):
    return 6 * v - u - 7

W_point = np.array([4.0, 4.0])
path_u = [W_point[0]]
path_v = [W_point[1]]

alpha = 0.05
num_steps = 21

for i in range(num_steps):
    grad = np.array([Lu(W_point[0], W_point[1]), Lv(W_point[0], W_point[1])])
    W_point = W_point - alpha * grad
    path_u.append(W_point[0])
    path_v.append(W_point[1])

print("처음 위치:", (path_u[0], path_v[0]))
print("마지막 위치:", (path_u[-1], path_v[-1]))

### 함수/변수 사용법

```python
grad = np.array([Lu(W[0], W[1]), Lv(W[0], W[1])])
W = W - alpha * grad
```

- `grad`: 현재 위치에서의 기울기다.
- `alpha`: learning rate처럼 한 번에 이동하는 크기다.
- `W - alpha * grad`: gradient의 반대 방향으로 이동한다.

> 헷갈림 포인트:  
> gradient는 값이 가장 빠르게 증가하는 방향이다.  
> Loss를 줄이고 싶으므로 gradient 방향이 아니라 반대 방향으로 간다.

In [ ]:
plt.plot(path_u, path_v, "o-", label="gradient descent path")
plt.xlabel("u")
plt.ylabel("v")
plt.title("Gradient Descent Path")
plt.legend()
plt.show()

그래프 해석:

- 점들이 반복마다 이동한 위치다.
- 한 번에 정답으로 가는 것이 아니라 조금씩 이동한다.
- learning rate가 너무 크면 튈 수 있고, 너무 작으면 오래 걸린다.

## 7. 신장/체중 샘플 데이터 만들기

이제 실제 실습 데이터로 들어간다.

5명의 신장과 체중 데이터가 주어진다.

### 함수 사용법: `np.array()`

```python
np.array([[166, 58.7], [176, 75.7]])
```

- 리스트를 NumPy 2차원 배열로 만든다.
- 각 행은 한 사람의 데이터다.
- 0번째 열은 신장, 1번째 열은 체중이다.

In [ ]:
sampleData1 = np.array([
    [166, 58.7],
    [176.0, 75.7],
    [171.0, 62.1],
    [173.0, 70.4],
    [169.0, 60.1]
])

print(sampleData1)
print("shape:", sampleData1.shape)

출력 해석:

```text
shape = (5, 2)
```

- 5명 데이터다.
- 각 데이터는 신장과 체중 2개 값을 가진다.

## 8. 입력 x와 정답 y 분리

학습을 위해 입력과 정답을 분리한다.

### 슬라이싱 사용법

```python
x = sampleData1[:, 0]
y = sampleData1[:, 1]
```

- `:`는 모든 행을 의미한다.
- `0`은 첫 번째 열, 즉 신장이다.
- `1`은 두 번째 열, 즉 체중이다.

In [ ]:
x = sampleData1[:, 0]
y = sampleData1[:, 1]

print("x:", x)
print("y:", y)

> 필기 포인트:  
> 머신러닝에서는 입력과 정답을 명확히 나누는 것이 먼저다.  
> 이게 문제 정의의 코드 버전이라고 보면 된다.

## 9. 원본 데이터 산점도

산점도로 신장과 체중의 관계를 먼저 눈으로 확인한다.

### 함수 사용법: `plt.scatter()`

```python
plt.scatter(x, y, c="k", s=50)
```

- `x`: x축 데이터다.
- `y`: y축 데이터다.
- `c`: 색상이다. `"k"`는 검정색이다.
- `s`: 점 크기다.

In [ ]:
plt.scatter(x, y, c="k", s=50)
plt.xlabel("x: height(cm)")
plt.ylabel("y: weight(kg)")
plt.title("Height and Weight")
plt.show()

그래프 해석:

- 점 하나가 한 사람의 신장과 체중이다.
- 점들이 대체로 오른쪽 위 방향으로 가면 신장이 클수록 체중이 증가하는 경향이 있다고 볼 수 있다.
- 이 점들을 가장 잘 대표하는 직선을 찾는 것이 이번 실습의 목표다.

## 10. 평균 빼기 전처리

경사하강법은 값의 크기가 너무 크면 학습이 불안정할 수 있다.

그래서 각 데이터에서 평균을 빼서 중심을 0 근처로 옮긴다.

```python
X = x - x.mean()
Y = y - y.mean()
```

### 함수 사용법: `.mean()`

```python
x.mean()
```

- 배열의 평균을 계산한다.
- 평균을 빼면 데이터 중심이 0으로 이동한다.

In [ ]:
X_np = x - x.mean()
Y_np = y - y.mean()

print("X:", X_np)
print("Y:", Y_np)
print("X mean:", X_np.mean())
print("Y mean:", Y_np.mean())

전처리 효과:

- 값의 중심이 0 근처로 이동한다.
- 데이터의 상대적인 관계는 유지된다.
- 경사하강법이 조금 더 안정적으로 작동하기 쉽다.

> 주의:  
> 평균을 뺐다고 데이터 의미가 사라지는 것은 아니다.  
> 원래 값에서 중심만 옮긴 것이다.

In [ ]:
plt.scatter(X_np, Y_np, c="k", s=50)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Centered Height and Weight")
plt.show()

그래프 해석:

- 원본 산점도와 모양은 비슷하다.
- 단지 좌표의 중심이 0 근처로 이동했다.
- 모델 학습에는 보통 이런 스케일 조정이 도움이 된다.

## 11. NumPy 배열을 Tensor로 변환

PyTorch로 학습하려면 NumPy 배열을 Tensor로 바꿔야 한다.

### 함수 사용법

```python
torch.tensor(array).float()
```

- `torch.tensor(array)`: NumPy 배열이나 리스트를 Tensor로 변환한다.
- `.float()`: float32 타입으로 바꾼다.
- PyTorch 모델 학습에서는 float 타입을 자주 사용한다.

In [ ]:
X = torch.tensor(X_np).float()
Y = torch.tensor(Y_np).float()

print("X:", X)
print("Y:", Y)
print("X dtype:", X.dtype)
print("Y dtype:", Y.dtype)

> 기억할 점:  
> 입력 X와 정답 Y는 모두 Tensor여야 PyTorch 연산과 자동 미분 흐름에 자연스럽게 들어간다.

## 12. 학습할 파라미터 W와 B 만들기

이제 모델이 학습할 파라미터를 만든다.

```python
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()
```

- `W`: 직선의 기울기다.
- `B`: 직선의 절편이다.
- `requires_grad=True`: PyTorch가 이 값의 gradient를 계산하도록 설정한다.

> 3강 연결:  
> `requires_grad=True`가 있어야 `loss.backward()` 후 `.grad`에 gradient가 저장된다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

print("W:", W)
print("B:", B)
print("W.requires_grad:", W.requires_grad)
print("B.requires_grad:", B.requires_grad)

## 13. 예측 함수 pred 만들기

선형 모델의 예측 함수는 다음이다.

```text
Yp = W * X + B
```

### 함수 사용법

```python
def pred(X):
    return W * X + B
```

- `X`: 입력 Tensor다.
- `W`, `B`: 밖에서 정의한 학습 파라미터다.
- 반환값 `Yp`: 현재 W와 B로 계산한 예측값이다.

In [ ]:
def pred(X):
    return W * X + B

Yp = pred(X)

print("Yp:", Yp)

> 헷갈림 포인트:  
> 여기서 `pred`는 모델 역할을 하는 함수다.  
> 뒤에서 `nn.Module`을 쓰면 이 함수 역할을 모델 클래스가 하게 된다.

## 14. MSE 손실 함수 만들기

예측값과 정답이 얼마나 다른지 계산해야 한다.

이번 실습에서는 MSE를 사용한다.

```text
MSE = mean((Yp - Y)²)
```

### 함수 사용법

```python
def mse(Yp, Y):
    loss = ((Yp - Y) ** 2).mean()
    return loss
```

- `Yp`: 예측값이다.
- `Y`: 정답값이다.
- `** 2`: 오차를 제곱한다.
- `.mean()`: 전체 평균을 낸다.

In [ ]:
def mse(Yp, Y):
    loss = ((Yp - Y) ** 2).mean()
    return loss

loss = mse(Yp, Y)

print("loss:", loss)

> 필기 포인트:  
> Loss는 모델의 현재 성적표다.  
> Loss가 작아질수록 예측이 정답에 가까워진다.

## 15. backward()로 경사 계산

이제 손실을 기준으로 W와 B가 얼마나 책임이 있는지 계산한다.

### 함수 사용법

```python
loss.backward()
```

- 계산 그래프를 거꾸로 따라간다.
- `requires_grad=True`인 Tensor의 gradient를 계산한다.
- 결과는 각 Tensor의 `.grad`에 저장된다.

In [ ]:
loss.backward()

print("W.grad:", W.grad)
print("B.grad:", B.grad)

출력 해석:

- `W.grad`: W를 어느 방향으로 얼마나 수정해야 하는지 알려주는 값이다.
- `B.grad`: B를 어느 방향으로 얼마나 수정해야 하는지 알려주는 값이다.

> 강의 핵심:  
> 역전파는 출력의 오류를 입력 방향으로 되돌리며 각 파라미터의 책임을 나누는 과정이다.

## 16. 잘못된 파라미터 수정 방식 확인

직관적으로는 다음처럼 쓰고 싶다.

```python
W -= lr * W.grad
B -= lr * B.grad
```

하지만 `requires_grad=True`인 leaf Tensor를 직접 in-place로 수정하면 에러가 난다.

이 에러를 일부러 확인해본다.

In [ ]:
lr = 0.001

try:
    W -= lr * W.grad
    B -= lr * B.grad
except RuntimeError as e:
    print("에러 발생:")
    print(e)

에러 이유:

- `W`와 `B`는 gradient 추적 대상이다.
- PyTorch는 이 값들이 계산 그래프에서 어떻게 쓰이는지 추적하고 있다.
- 그런데 `-=`처럼 원본을 직접 바꾸면 추적 흐름이 꼬일 수 있다.

그래서 파라미터 수정은 gradient 추적을 끄고 해야 한다.

## 17. torch.no_grad()로 올바르게 수정하기

파라미터 업데이트는 `torch.no_grad()` 안에서 수행한다.

### 함수 사용법

```python
with torch.no_grad():
    W -= lr * W.grad
```

- 이 블록 안에서는 계산 그래프를 만들지 않는다.
- 학습 파라미터를 안전하게 직접 수정할 수 있다.

업데이트 후에는 gradient를 초기화한다.

```python
W.grad.zero_()
B.grad.zero_()
```

In [ ]:
with torch.no_grad():
    W -= lr * W.grad
    B -= lr * B.grad

W.grad.zero_()
B.grad.zero_()

print("W:", W)
print("B:", B)
print("W.grad:", W.grad)
print("B.grad:", B.grad)

> 시험 포인트:  
> 직접 업데이트할 때는 `torch.no_grad()`가 필요하다.  
> gradient는 누적되므로 업데이트 후 `zero_()`로 초기화해야 한다.

## 18. 반복 학습 준비

이제 위 과정을 500번 반복한다.

초기화할 것:

- `W`, `B`: 다시 1.0에서 시작한다.
- `num_epochs`: 반복 횟수다.
- `lr`: learning rate다.
- `history`: epoch와 loss를 기록할 배열이다.

### 함수 사용법: `np.zeros()`

```python
np.zeros((0, 2))
```

- 처음에는 행이 0개, 열이 2개인 빈 배열을 만든다.
- 나중에 `[epoch, loss]`를 계속 붙인다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

num_epochs = 500
lr = 0.001

history = np.zeros((0, 2))

print("초기 W:", W.item())
print("초기 B:", B.item())
print("history shape:", history.shape)

## 19. 직접 구현한 학습 루프

학습 루프의 핵심 순서는 다음이다.

```text
예측 계산 → 손실 계산 → 경사 계산 → 파라미터 수정 → 경사 초기화 → 기록
```

이 구조가 PyTorch 학습 코드의 기본 뼈대다.

In [ ]:
for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)

    loss.backward()

    with torch.no_grad():
        W -= lr * W.grad
        B -= lr * B.grad

    W.grad.zero_()
    B.grad.zero_()

    if epoch % 10 == 0:
        item = np.array([epoch, loss.item()])
        history = np.vstack((history, item))

print("최종 W:", W.item())
print("최종 B:", B.item())
print("초기 loss:", history[0, 1])
print("최종 loss:", history[-1, 1])

### 함수 사용법 정리

```python
loss.item()
```

- 원소 하나짜리 Tensor에서 Python 숫자를 꺼낸다.
- loss 기록에 자주 사용한다.

```python
np.vstack((history, item))
```

- 기존 배열 아래에 새 행을 붙인다.
- 여기서는 `[epoch, loss]` 기록을 누적한다.

> 필기 포인트:  
> 반복하면서 Loss가 줄어들면 모델이 학습되고 있다는 뜻이다.

## 20. 학습 곡선 시각화

Loss가 반복에 따라 어떻게 줄어드는지 그래프로 본다.

### 함수 사용법: `plt.plot()`

```python
plt.plot(history[:, 0], history[:, 1])
```

- x축은 epoch다.
- y축은 loss다.
- loss가 내려가면 학습이 진행되는 것이다.

In [ ]:
plt.plot(history[:, 0], history[:, 1], "b")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss")
plt.show()

그래프 해석:

- 초반에는 Loss가 빠르게 감소한다.
- 뒤로 갈수록 완만하게 줄어든다.
- 경사하강법이 점점 낮은 Loss 지점으로 이동하고 있다는 뜻이다.

## 21. 가공된 데이터에서 학습 직선 확인

학습된 W와 B가 만든 직선을 산점도 위에 그린다.

이번 그래프는 평균을 뺀 `X`, `Y` 기준이다.

In [ ]:
X_range = torch.tensor([X.min(), X.max()]).float()
Y_range = pred(X_range)

plt.scatter(X, Y, c="k", s=50)
plt.plot(X_range.data, Y_range.data, linewidth=2)
plt.xlabel("X")
plt.ylabel("Y")
plt.title("Fitted Line after Centering")
plt.show()

그래프 해석:

- 검은 점은 실제 데이터다.
- 선은 모델이 학습한 관계다.
- 점들의 흐름을 잘 대표하면 학습이 잘 된 것이다.

## 22. 원본 스케일로 다시 보기

전처리할 때 평균을 뺐으므로, 원본 단위로 보려면 평균을 다시 더한다.

```python
x_range = X_range + x.mean()
yp_range = Y_range + y.mean()
```

- x축은 다시 신장 cm 단위가 된다.
- y축은 다시 체중 kg 단위가 된다.

In [ ]:
x_range = X_range + x.mean()
yp_range = Y_range + y.mean()

plt.scatter(x, y, c="k", s=50)
plt.plot(x_range, yp_range.data, linewidth=2)
plt.xlabel("height(cm)")
plt.ylabel("weight(kg)")
plt.title("Fitted Line in Original Scale")
plt.show()

> 기억할 점:  
> 모델 학습은 전처리된 값으로 해도, 결과 해석은 원본 단위로 다시 보여줘야 이해하기 쉽다.

## 23. Optimizer 사용하기

지금까지는 직접 다음처럼 업데이트했다.

```python
W -= lr * W.grad
B -= lr * B.grad
```

이제 PyTorch의 Optimizer가 이 일을 대신하게 한다.

### 함수 사용법: `optim.SGD()`

```python
optimizer = optim.SGD([W, B], lr=lr)
```

- `[W, B]`: Optimizer가 관리할 파라미터 목록이다.
- `lr`: learning rate다.
- `SGD`: Stochastic Gradient Descent 계열 Optimizer다.

In [ ]:
W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

optimizer = optim.SGD([W, B], lr=lr)

history_optimizer = np.zeros((0, 2))

print(optimizer)

Optimizer를 쓰면 업데이트 코드가 더 간단해진다.

핵심은 두 함수다.

```python
optimizer.step()
optimizer.zero_grad()
```

- `step()`: 파라미터를 수정한다.
- `zero_grad()`: Optimizer가 관리하는 파라미터들의 gradient를 0으로 만든다.

## 24. Optimizer 학습 루프

Optimizer를 사용하면 `torch.no_grad()` 안에서 직접 수정하지 않아도 된다.

In [ ]:
for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)

    loss.backward()

    optimizer.step()
    optimizer.zero_grad()

    if epoch % 10 == 0:
        item = np.array([epoch, loss.item()])
        history_optimizer = np.vstack((history_optimizer, item))

print("최종 W:", W.item())
print("최종 B:", B.item())
print("초기 loss:", history_optimizer[0, 1])
print("최종 loss:", history_optimizer[-1, 1])

코드 흐름:

```text
Yp = pred(X)        → 예측
loss = mse(Yp, Y)  → 손실
loss.backward()    → 경사 계산
optimizer.step()   → 파라미터 수정
optimizer.zero_grad() → 경사 초기화
```

> 시험 포인트:  
> `loss.backward()`와 `optimizer.step()`의 순서가 중요하다.  
> gradient를 계산한 다음에 step으로 수정한다.

In [ ]:
plt.plot(history_optimizer[:, 0], history_optimizer[:, 1])
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("Training Loss with Optimizer")
plt.show()

그래프 해석:

- 직접 업데이트한 경우와 비슷하게 Loss가 감소한다.
- Optimizer를 쓰면 코드가 더 간결하고, momentum 같은 옵션도 쉽게 추가할 수 있다.

## 25. Momentum 적용하기

Momentum은 이전 이동 방향의 관성을 반영하는 방식이다.

강의에서는 Optimizer가 파라미터 업데이트를 코칭하는 역할이라고 설명했다.

### 함수 사용법

```python
optim.SGD([W, B], lr=lr, momentum=0.9)
```

- `momentum=0.9`: 이전 방향을 어느 정도 반영할지 정한다.
- 경사가 계속 같은 방향이면 더 빠르게 이동할 수 있다.

In [ ]:
history_default = history_optimizer.copy()

W = torch.tensor(1.0, requires_grad=True).float()
B = torch.tensor(1.0, requires_grad=True).float()

optimizer = optim.SGD([W, B], lr=lr, momentum=0.9)
history_momentum = np.zeros((0, 2))

for epoch in range(num_epochs):
    Yp = pred(X)
    loss = mse(Yp, Y)

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if epoch % 10 == 0:
        item = np.array([epoch, loss.item()])
        history_momentum = np.vstack((history_momentum, item))

print("Momentum 최종 loss:", history_momentum[-1, 1])

> 기억할 점:  
> Momentum은 매번 완전히 새 방향으로 움직이는 것이 아니라, 이전에 가던 방향의 흐름을 어느 정도 유지한다.  
> 그래서 경사하강법의 진동을 줄이고 수렴을 빠르게 만들 수 있다.

In [ ]:
plt.plot(history_default[:, 0], history_default[:, 1], label="SGD")
plt.plot(history_momentum[:, 0], history_momentum[:, 1], label="SGD + momentum")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("SGD vs Momentum")
plt.legend()
plt.show()

그래프 해석:

- Momentum을 적용한 선이 더 빠르게 내려가면 학습 가속 효과가 있다고 볼 수 있다.
- 복잡한 이론을 직접 구현하지 않아도 Optimizer 옵션 하나로 학습 방식을 바꿀 수 있다.

## 26. 국소 최적해 개념 보기

강의에서는 학습이 잘 안 되는 이유 중 하나로 국소 최적해를 설명했다.

```text
전역 최적해: 전체 지형에서 가장 낮은 지점
국소 최적해: 주변에서는 낮지만 전체에서는 최고가 아닌 지점
```

초기 파라미터가 어디서 시작하느냐에 따라 도착지가 달라질 수 있다.

In [ ]:
def f_local(x):
    return x * (x + 1) * (x + 2) * (x - 2)

x_curve = np.arange(-3, 2.7, 0.05)
y_curve = f_local(x_curve)

plt.plot(x_curve, y_curve)
plt.xlabel("x")
plt.ylabel("f(x)")
plt.title("Local Minimum Example")
plt.show()

그래프 해석:

- 손실 지형이 울퉁불퉁하면 여러 낮은 지점이 생길 수 있다.
- 단순 경사하강법은 현재 위치 기준으로만 내려가기 때문에 국소 최적해에 갇힐 수 있다.
- 그래서 Optimizer, momentum, 학습률 조절 같은 방법이 중요해진다.

## 27. 활성화 함수가 필요한 이유

강의 자료에서 가장 중요한 문장 중 하나다.

```text
Linear + Linear = Linear
```

즉, 선형 함수만 여러 층 쌓아도 결과는 여전히 선형 함수다.

그래서 신경망이 복잡한 패턴을 배우려면 중간에 비선형 함수가 필요하다.  
그 역할을 하는 것이 활성화 함수다.

```text
Linear → Activation → Linear → Activation
```

> 시험 포인트:  
> 활성화 함수가 없으면 아무리 깊게 쌓아도 결국 하나의 선형 모델과 비슷해진다.

In [ ]:
activation_summary = {
    "Sigmoid": "0과 1 사이로 출력되며 확률처럼 해석할 수 있다",
    "Tanh": "-1과 1 사이로 출력되며 zero-centered 특성이 있다",
    "ReLU": "음수는 0, 양수는 그대로 통과시키며 딥러닝에서 자주 사용된다"
}

for key, value in activation_summary.items():
    print(f"{key}: {value}")

## 28. 대표 활성화 함수 그래프

Sigmoid, Tanh, ReLU를 그래프로 비교한다.

### 함수 사용법

```python
torch.sigmoid(x)
torch.tanh(x)
torch.relu(x)
```

- `torch.sigmoid(x)`: 0과 1 사이 값으로 바꾼다.
- `torch.tanh(x)`: -1과 1 사이 값으로 바꾼다.
- `torch.relu(x)`: 음수는 0, 양수는 그대로 둔다.

In [ ]:
x_act = torch.linspace(-3, 3, 100)

sigmoid = torch.sigmoid(x_act)
tanh = torch.tanh(x_act)
relu = torch.relu(x_act)

plt.plot(x_act, sigmoid, label="Sigmoid")
plt.plot(x_act, tanh, label="Tanh")
plt.plot(x_act, relu, label="ReLU")
plt.xlabel("x")
plt.ylabel("activation(x)")
plt.title("Activation Functions")
plt.legend()
plt.show()

그래프 해석:

- Sigmoid는 S자 모양이고 양 끝에서 gradient가 작아진다.
- Tanh도 S자 계열이지만 출력 중심이 0에 가깝다.
- ReLU는 양수 영역에서 기울기가 유지되어 깊은 신경망에서 자주 쓰인다.
- 단, 음수 영역에서는 출력이 0이라 Dying ReLU 문제가 생길 수 있다.

## 29. MLP 구조 맛보기

신경망은 보통 `Linear`와 `Activation`을 반복한다.

### 함수 사용법: `nn.Sequential()`

```python
nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)
```

- 여러 Layer를 순서대로 묶는다.
- 입력이 첫 Layer를 지나고, 그 결과가 다음 Layer로 넘어간다.

### 함수 사용법: `nn.Linear()`

```python
nn.Linear(in_features, out_features)
```

- 입력 feature 수를 출력 feature 수로 바꾸는 선형 Layer다.

In [ ]:
model_example = nn.Sequential(
    nn.Linear(1, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

print(model_example)

> 필기 포인트:  
> `Linear + Activation` 반복 구조는 MLP, CNN, Transformer에도 공통적으로 들어가는 기본 DNA처럼 볼 수 있다.

## 30. Bias-Variance 개념

강의 후반부에서는 모델 용량과 Bias-Variance를 다뤘다.

| 개념 | 의미 |
|---|---|
| High Bias | 모델이 너무 단순해서 패턴을 못 배운다 |
| High Variance | 모델이 너무 복잡해서 훈련 데이터를 과하게 외운다 |
| Underfitting | train/validation 오차가 둘 다 높다 |
| Overfitting | train 오차는 낮지만 validation 오차가 높다 |

> 기억할 점:  
> 목표는 너무 단순하지도, 너무 복잡하지도 않은 적절한 균형점이다.

In [ ]:
bias_variance = {
    "High Bias": "모델이 너무 단순해서 패턴을 잘 못 배운다",
    "High Variance": "모델이 너무 복잡해서 훈련 데이터에 과하게 민감하다",
    "Underfitting": "Train loss와 Validation loss가 모두 높은 상태다",
    "Overfitting": "Train loss는 낮지만 Validation loss가 높은 상태다"
}

for key, value in bias_variance.items():
    print(f"{key}: {value}")

## 31. Bias-Variance 실습 데이터 만들기

사인파에 노이즈를 섞은 회귀 데이터를 만든다.

### 함수 사용법

```python
torch.linspace(start, end, steps)
```

- 시작값부터 끝값까지 일정 간격으로 Tensor를 만든다.

```python
unsqueeze(1)
```

- 1번 위치에 크기 1인 차원을 추가한다.
- `[N]`을 `[N, 1]` 형태로 바꾼다.

```python
torch.randn_like(x)
```

- x와 같은 shape의 정규분포 난수를 만든다.

In [ ]:
torch.manual_seed(0)

N = 600

x_sin = torch.linspace(-3 * math.pi, 3 * math.pi, N).unsqueeze(1)
y_sin = torch.sin(x_sin) + 0.2 * torch.randn_like(x_sin)

print("x_sin shape:", x_sin.shape)
print("y_sin shape:", y_sin.shape)

In [ ]:
plt.scatter(x_sin, y_sin, s=8)
plt.xlabel("x")
plt.ylabel("y")
plt.title("Sine Data with Noise")
plt.show()

그래프 해석:

- 전체적으로 사인파 모양이다.
- 노이즈가 섞여 있어서 점들이 완벽한 곡선 위에 있지 않다.
- 모델이 너무 복잡하면 이 노이즈까지 외울 위험이 있다.

## 32. Train / Validation / Test 분할

데이터는 학습용, 검증용, 테스트용으로 나눈다.

강의에서는 시험 공부 비유로 설명했다.

```text
Train: 교과서로 공부
Validation: 모의고사로 튜닝
Test: 실제 수능으로 최종 평가
```

### 함수 사용법: `train_test_split()`

```python
train_test_split(X, y, test_size=0.4, random_state=42)
```

- `X`: 입력 데이터다.
- `y`: 정답 데이터다.
- `test_size`: 떼어낼 데이터 비율이다.
- `random_state`: 랜덤 결과를 고정한다.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    x_sin.numpy(),
    y_sin.numpy(),
    test_size=0.4,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

> 시험 포인트:  
> Test 데이터는 최종 평가용이다.  
> 학습이나 튜닝에 Test를 사용하면 데이터 누수다.

## 33. 작은 모델과 큰 모델 만들기

모델 용량을 비교하기 위해 작은 MLP와 큰 MLP를 만든다.

### 함수 사용법

```python
make_mlp(hidden)
```

- `hidden`: 은닉층 뉴런 수다.
- 작으면 모델 용량이 낮다.
- 크면 모델 용량이 높다.

구조는 다음과 같다.

```text
Linear → ReLU → Linear → ReLU → Linear
```

In [ ]:
def make_mlp(hidden):
    return nn.Sequential(
        nn.Linear(1, hidden),
        nn.ReLU(),
        nn.Linear(hidden, hidden),
        nn.ReLU(),
        nn.Linear(hidden, 1)
    )

small = make_mlp(hidden=8)
big = make_mlp(hidden=128)

print("Small model:")
print(small)

print("\nBig model:")
print(big)

해석:

- `small`: 파라미터 수가 적어 단순한 모델이다.
- `big`: 파라미터 수가 많아 복잡한 모델이다.
- 큰 모델은 더 잘 맞출 수 있지만, 노이즈까지 외울 위험도 있다.

## 34. 학습 함수 만들기

Train loss와 Validation loss를 함께 기록하는 함수를 만든다.

### 함수 사용법

```python
train(model, Xtr, ytr, Xva, yva, epochs=400, lr=1e-3)
```

- `model`: 학습할 모델이다.
- `Xtr`, `ytr`: 훈련 입력과 정답이다.
- `Xva`, `yva`: 검증 입력과 정답이다.
- `epochs`: 반복 횟수다.
- `lr`: learning rate다.

함수 내부 핵심은 다음이다.

```text
model.train() → 학습 모드
loss.backward() → 경사 계산
optimizer.step() → 파라미터 수정
model.eval() + torch.no_grad() → 검증
```

In [ ]:
def train(model, Xtr, ytr, Xva, yva, epochs=400, lr=1e-3):
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    tr_hist = []
    va_hist = []

    for ep in range(epochs):
        model.train()
        opt.zero_grad()

        pred_value = model(Xtr)
        loss = loss_fn(pred_value, ytr)

        loss.backward()
        opt.step()

        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xva), yva).item()

        tr_hist.append(loss.item())
        va_hist.append(val_loss)

    return tr_hist, va_hist

### 함수 사용법 정리

```python
model.train()
```

- Dropout, BatchNorm 같은 Layer가 학습 모드로 작동하게 한다.

```python
model.eval()
```

- 평가 모드로 바꾼다.

```python
with torch.no_grad():
```

- 평가할 때 gradient 계산을 끈다.

```python
nn.MSELoss()
```

- 회귀 문제에서 예측값과 정답의 평균 제곱 오차를 계산한다.

## 35. 작은 모델과 큰 모델 학습

두 모델을 같은 데이터로 학습하고 train/validation loss를 비교한다.

In [ ]:
tr_s, va_s = train(small, X_train, y_train, X_val, y_val, epochs=400)
tr_b, va_b = train(big, X_train, y_train, X_val, y_val, epochs=400)

print("small train loss:", tr_s[-1])
print("small val loss:", va_s[-1])

print("big train loss:", tr_b[-1])
print("big val loss:", va_b[-1])

결과를 볼 때는 train loss만 보면 안 된다.

- train loss만 낮고 val loss가 높으면 과적합일 수 있다.
- train loss와 val loss가 모두 높으면 과소적합일 수 있다.

## 36. Bias-Variance 학습 곡선 비교

학습 곡선을 한 그래프에 그려서 비교한다.

In [ ]:
plt.plot(tr_s, label="small-train")
plt.plot(va_s, label="small-val")
plt.plot(tr_b, label="big-train")
plt.plot(va_b, label="big-val")

plt.xlabel("epoch")
plt.ylabel("MSE Loss")
plt.title("Bias-Variance")
plt.legend()
plt.show()

그래프 해석:

- 작은 모델은 표현력이 부족해서 train/val 모두 충분히 낮아지지 않을 수 있다.
- 큰 모델은 train loss를 더 낮출 수 있지만 validation과 gap이 생길 수 있다.
- 이 gap이 커지면 과적합을 의심한다.

> 기억할 점:  
> 모델을 평가할 때는 train 성능보다 validation/test 성능이 더 중요하다.

## 37. Test MSE 계산

Test 데이터는 최종 평가에만 사용한다.

### 함수 사용법

```python
test_mse(model, X, y)
```

- 모델을 평가 모드로 바꾼다.
- `torch.no_grad()` 안에서 예측한다.
- MSELoss를 계산한다.

In [ ]:
def test_mse(model, X, y):
    model.eval()
    with torch.no_grad():
        return nn.MSELoss()(model(X), y).item()

print("Small Test MSE:", test_mse(small, X_test, y_test))
print("Big Test MSE:", test_mse(big, X_test, y_test))

> 데이터 누수 주의:  
> Test 결과를 보고 모델 구조나 하이퍼파라미터를 계속 바꾸면, Test도 사실상 검증 데이터처럼 써버리는 것이다.  
> 그러면 최종 성능 평가가 부정확해진다.

## 38. 성능 평가 지표 큰 그림

강의 마지막 부분에서는 평가 지표도 정리했다.

문제 유형에 따라 지표가 다르다.

| 문제 | 대표 지표 |
|---|---|
| 회귀 | MSE, MAE, R² Score |
| 분류 | Accuracy, Precision, Recall, F1, ROC-AUC |

이번 실습의 신장/체중 예측과 사인파 예측은 회귀 문제라서 MSE를 사용했다.

> 필기 포인트:  
> 문제 정의가 평가 지표를 결정한다.  
> 회귀는 오차를 줄이는 것이 중요하고, 분류는 맞춘 비율이나 양성 탐지 성능이 중요하다.

In [ ]:
metrics = {
    "MSE": "Mean Squared Error, 회귀에서 평균 제곱 오차",
    "MAE": "Mean Absolute Error, 회귀에서 평균 절대 오차",
    "R2": "결정계수, 회귀 모델의 설명력",
    "Accuracy": "전체 중 맞춘 비율",
    "Precision": "양성이라고 예측한 것 중 실제 양성 비율",
    "Recall": "실제 양성 중 찾아낸 비율",
    "F1": "Precision과 Recall의 조화평균",
    "ROC-AUC": "임계값 전체에서의 분류 성능"
}

for key, value in metrics.items():
    print(f"{key}: {value}")

## 39. 주요 함수 / 변수 / 약어 정리

| 이름 | 뜻 | 어떻게 쓰는지 |
|---|---|---|
| `X` | 입력 데이터 | 모델에 넣는 값 |
| `Y` | 정답 데이터 | 모델이 맞혀야 하는 값 |
| `Yp` | prediction | 예측값 |
| `W` | weight | 선형 모델의 기울기 |
| `B` | bias | 선형 모델의 절편 |
| `loss` | 손실 | 예측이 얼마나 틀렸는지 |
| `grad` | gradient | 파라미터 수정 방향 |
| `lr` | learning rate | 한 번에 이동하는 크기 |
| `epoch` | 반복 단위 | 전체 학습 반복 횟수 |
| `requires_grad` | 자동 미분 추적 | `requires_grad=True` |
| `backward()` | 역전파 | `loss.backward()` |
| `zero_()` | gradient 초기화 | `W.grad.zero_()` |
| `torch.no_grad()` | gradient 추적 중지 | 파라미터 직접 수정이나 평가에 사용 |
| `optim.SGD` | SGD Optimizer | `optim.SGD(params, lr=...)` |
| `optimizer.step()` | 파라미터 수정 | gradient 기준으로 업데이트 |
| `optimizer.zero_grad()` | gradient 초기화 | Optimizer가 관리하는 grad를 0으로 |
| `momentum` | 관성 옵션 | 이전 이동 방향을 반영 |
| `MSE` | 평균 제곱 오차 | `((Yp - Y)**2).mean()` |
| `nn.Linear` | 선형 Layer | `nn.Linear(in, out)` |
| `nn.ReLU` | 활성화 함수 | 음수는 0, 양수는 그대로 |
| `nn.Sequential` | Layer 순서 묶음 | 여러 Layer를 한 모델처럼 사용 |
| `model.train()` | 학습 모드 | 학습할 때 호출 |
| `model.eval()` | 평가 모드 | 검증/테스트할 때 호출 |
| `train_test_split` | 데이터 분할 | train/val/test 나눌 때 사용 |

## 40. 시험용 요약

```text
학습 루프 = Forward → Loss → Backward → Update
```

꼭 기억할 것:

- 모든 딥러닝의 시작은 선형 모델이다.
- 선형회귀 기본식은 `Yp = W * X + B`다.
- 문제 정의가 입력, 정답, 손실 함수, 평가 지표를 결정한다.
- 경사하강법은 Loss를 줄이는 방향으로 파라미터를 조금씩 수정하는 방법이다.
- Learning Rate는 한 번에 이동하는 크기다.
- `requires_grad=True`가 있어야 W와 B의 gradient를 계산할 수 있다.
- MSE는 예측값과 정답의 차이를 제곱한 뒤 평균내는 손실 함수다.
- `loss.backward()`는 역전파를 수행해 `.grad`에 gradient를 저장한다.
- 직접 파라미터를 수정할 때는 `torch.no_grad()`가 필요하다.
- gradient는 누적되므로 매 반복마다 초기화해야 한다.
- `optimizer.step()`은 파라미터 업데이트다.
- `optimizer.zero_grad()`는 gradient 초기화다.
- Momentum은 이전 이동 방향의 관성을 반영해 학습을 빠르고 안정적으로 만들 수 있다.
- 선형 함수만 여러 층 쌓아도 결과는 여전히 선형이다.
- 활성화 함수는 신경망에 비선형성을 추가한다.
- ReLU는 딥러닝에서 자주 쓰이는 활성화 함수다.
- Bias가 크면 과소적합, Variance가 크면 과적합으로 이어질 수 있다.
- Train은 학습, Validation은 튜닝, Test는 최종 평가에 사용한다.
- Test 데이터가 학습이나 튜닝에 섞이면 데이터 누수다.